In [39]:
!pip install -q streamlit bcrypt pyjwt pyngrok
print("All required dependencies installed successfully.")


All required dependencies installed successfully.


In [40]:
import os

try:
    from google.colab import userdata
    COLAB_AVAILABLE = True
except ImportError:
    COLAB_AVAILABLE = False

def check_secret(name: str, required: bool = True) -> bool:
    val = None
    if COLAB_AVAILABLE:
        try:
            val = userdata.get(name)
        except Exception:
            val = None
    if not val:
        val = os.environ.get(name)

    if val:
        print(f" [CONFIGURED] {name} is present.")
        return True
    else:
        if required:
            print(f" [MISSING]    {name} is NOT set in Colab Secrets! (Required)")
        else:
            print(f" [OPTIONAL]   {name} is not set; admin login stays disabled until it is set.")
        return False

print("--- Validating Colab Secrets Configuration ---")
s1 = check_secret("JWT_SECRET", required=True)
s2 = check_secret("NGROK_AUTHTOKEN", required=True)
s3 = check_secret("EMAIL_ADDRESS", required=True)
s4 = check_secret("EMAIL_PASSWORD", required=True)
s5 = check_secret("ADMIN_PASSWORD", required=False)

if not (s1 and s2 and s3 and s4):
    print("\n[!] WARNING: Please configure missing secrets in the Colab Secrets (Key icon) tab to test all features.")
else:
    print("\nAll required secrets are successfully validated!")


--- Validating Colab Secrets Configuration ---
 [CONFIGURED] JWT_SECRET is present.
 [CONFIGURED] NGROK_AUTHTOKEN is present.
 [CONFIGURED] EMAIL_ADDRESS is present.
 [CONFIGURED] EMAIL_PASSWORD is present.
 [CONFIGURED] ADMIN_PASSWORD is present.

All required secrets are successfully validated!


In [41]:
import os
import sqlite3

# Define paths
APP_DIR = "/content/franchiseops_ai"
DATA_DIR = os.path.join(APP_DIR, "data")
DB_PATH = os.path.join(DATA_DIR, "users.db")

os.makedirs(DATA_DIR, exist_ok=True)

def init_db(db_path: str = DB_PATH) -> None:
    """Idempotent SQLite initialization creating tables and indexes."""
    conn = sqlite3.connect(db_path)
    with conn:
        # Users Table
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                email TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL,
                question_id TEXT NOT NULL,
                answer_hash TEXT NOT NULL,
                role TEXT NOT NULL DEFAULT 'user',
                created_at TEXT NOT NULL,
                status TEXT NOT NULL DEFAULT 'active',
                token_version INTEGER NOT NULL DEFAULT 1
            )
        """)
        # OTP Recovery Table
        conn.execute("""
            CREATE TABLE IF NOT EXISTS password_reset_otps (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                otp_hash TEXT NOT NULL,
                created_at TEXT NOT NULL,
                expires_at TEXT NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0,
                consumed INTEGER NOT NULL DEFAULT 0,
                purpose TEXT NOT NULL DEFAULT 'password_reset'
            )
        """)
        # Indexes for fast querying
        conn.execute("CREATE INDEX IF NOT EXISTS idx_users_username ON users(username)")
        conn.execute("CREATE INDEX IF NOT EXISTS idx_users_email ON users(email)")
        conn.execute("CREATE INDEX IF NOT EXISTS idx_otps_email ON password_reset_otps(email)")
    conn.close()
    print(f"SQLite database initialized idempotently at: {db_path}")

init_db()


SQLite database initialized idempotently at: /content/franchiseops_ai/data/users.db


In [42]:
import re
import hmac
import hashlib
import bcrypt
from typing import Optional

EMAIL_REGEX = re.compile(r"^[a-zA-Z0-9._%+-]{2,}@[a-zA-Z0-9.-]{2,}\.[a-zA-Z]{2,}$")
USERNAME_REGEX = re.compile(r"^[a-zA-Z0-9_]{3,30}$")

SECURITY_QUESTIONS = {
    "Q1": "What was the name of your first school?",
    "Q2": "What was the name of your first pet?",
    "Q3": "What city were you born in?",
    "Q4": "What was your childhood nickname?"
}

def validate_username(username: str) -> Optional[str]:
    u = username.strip()
    if not u:
        return "Username cannot be empty."
    if not USERNAME_REGEX.match(u):
        return "Username must be 3-30 characters and contain only letters, numbers, and underscores."
    return None

def validate_email(email: str) -> Optional[str]:
    e = email.strip()
    if not e:
        return "Email cannot be empty."
    if not EMAIL_REGEX.match(e):
        return "Invalid email. Must have at least 2 chars before @, valid domain, and 2+ char extension."
    return None

def validate_password_strength(password: str) -> Optional[str]:
    if len(password) < 8:
        return "Password must be at least 8 characters long."
    if not re.search(r"[A-Z]", password):
        return "Password must include at least one uppercase letter (A-Z)."
    if not re.search(r"[a-z]", password):
        return "Password must include at least one lowercase letter (a-z)."
    if not re.search(r"\d", password):
        return "Password must include at least one number (0-9)."
    if not re.search(r'[!@#$%^&*(),.?":{}|<>\-_=+]', password):
        return "Password must include at least one special character (!@#$%^&*...)."
    return None

def hash_password(plaintext: str) -> str:
    """Hash password with bcrypt and 12-round salt."""
    salt = bcrypt.gensalt(rounds=12)
    return bcrypt.hashpw(plaintext.encode("utf-8"), salt).decode("utf-8")

def verify_password(plaintext: str, hashed: str) -> bool:
    """Verify password against bcrypt hash."""
    try:
        return bcrypt.checkpw(plaintext.encode("utf-8"), hashed.encode("utf-8"))
    except Exception:
        return False

def hash_security_answer(answer: str) -> str:
    """Normalize (strip, lower) and hash security answer."""
    normalized = answer.strip().lower()
    salt = "FranchiseOpsAI_SecAnsSalt_2026"
    return hashlib.sha256(f"{salt}_{normalized}".encode("utf-8")).hexdigest()

def verify_security_answer(answer: str, stored_hash: str) -> bool:
    """Constant-time verification of security answer."""
    computed = hash_security_answer(answer)
    return hmac.compare_digest(computed, stored_hash)

print("Validation and cryptographic hashing utilities defined.")


Validation and cryptographic hashing utilities defined.


In [43]:
import smtplib
from email.message import EmailMessage
from typing import Tuple   # ← THIS was missing

def hash_otp(otp: str) -> str:
    salt = "FranchiseOpsAI_OTPSalt_2026"
    return hashlib.sha256(f"{salt}_{otp}".encode("utf-8")).hexdigest()

def send_otp_email(to_email: str, otp_code: str) -> Tuple[bool, str]:
    email_addr = None
    email_pass = None
    try:
        from google.colab import userdata
        email_addr = userdata.get("EMAIL_ADDRESS")
        email_pass = userdata.get("EMAIL_PASSWORD")
    except Exception:
        pass
    if not email_addr: email_addr = os.environ.get("EMAIL_ADDRESS")
    if not email_pass: email_pass = os.environ.get("EMAIL_PASSWORD")

    if not email_addr or not email_pass:
        return False, "Gmail credentials not configured in Colab Secrets."

    msg = EmailMessage()
    msg["Subject"] = f"FranchiseOps AI - Password Reset OTP: {otp_code}"
    msg["From"] = f"FranchiseOps AI Security <{email_addr}>"
    msg["To"] = to_email

    html_content = f"""
    <div style="font-family:sans-serif; max-width:480px; margin:auto; background:#ffffff; border:1px solid #DCE5DE; border-radius:12px; padding:28px;">
        <h2 style="color:#176B45; margin:0 0 8px 0;">FranchiseOps AI</h2>
        <p style="color:#68746D; font-size:14px; margin:0 0 20px 0;">Password Reset Security Service</p>
        <p>Your one-time password (OTP) for password recovery is:</p>
        <div style="background:#EBF5F0; border:1px dashed #18A86B; border-radius:8px; text-align:center; padding:16px; margin:20px 0;">
            <span style="font-size:32px; font-weight:800; letter-spacing:6px; color:#0B3D2E; font-family:monospace;">{otp_code}</span>
        </div>
        <p style="font-size:12px; color:#68746D;">&bull; Valid for <strong>5 minutes</strong>.<br>&bull; Maximum 5 verification attempts.<br>&bull; Never share this code with anyone.</p>
    </div>
    """
    msg.set_content(f"Your FranchiseOps AI OTP is {otp_code}. Valid for 5 minutes.")
    msg.add_alternative(html_content, subtype="html")

    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=15) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(email_addr, email_pass)
            server.send_message(msg)
        return True, "OTP successfully delivered to your email."
    except Exception as e:
        return False, f"SMTP delivery failed: {type(e).__name__}"

print("Gmail SMTP and OTP service configured.")


Gmail SMTP and OTP service configured.


In [44]:
def register_user(username: str, email: str, password: str, confirm_pw: str,
                  question_id: str, security_answer: str, db_path: str = DB_PATH) -> Tuple[bool, str]:
    u_err = validate_username(username)
    e_err = validate_email(email)
    p_err = validate_password_strength(password)

    if u_err: return False, u_err
    if e_err: return False, e_err
    if p_err: return False, p_err
    if password != confirm_pw: return False, "Passwords do not match."
    if not security_answer.strip(): return False, "Security answer cannot be empty."
    if question_id not in SECURITY_QUESTIONS: return False, "Invalid security question."

    norm_u = username.strip()
    norm_e = email.strip().lower()

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        existing = conn.execute(
            "SELECT username, email FROM users WHERE username = ? OR email = ?",
            (norm_u, norm_e)
        ).fetchone()
        if existing:
            if existing["username"].lower() == norm_u.lower():
                return False, "Username is already taken."
            return False, "This email is already registered."

        pw_h = hash_password(password)
        ans_h = hash_security_answer(security_answer)
        now_iso = datetime.now(timezone.utc).isoformat()

        with conn:
            conn.execute("""
                INSERT INTO users (username, email, password_hash, question_id, answer_hash, role, created_at, status, token_version)
                VALUES (?, ?, ?, ?, ?, 'user', ?, 'active', 1)
            """, (norm_u, norm_e, pw_h, question_id, ans_h, now_iso))
        return True, "Registration successful."
    finally:
        conn.close()

print("Signup registration backend logic defined.")


Signup registration backend logic defined.


In [45]:
# ── Standard Library ──────────────────────────────────────────────────────
import os
import re
import hmac
import hashlib
import smtplib
import sqlite3
import base64
import secrets
import random
import string
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage

# ── Type Hints (required for Python 3.9 / Google Colab) ──────────────────
from typing import Optional, Dict, Any, Tuple, List

print("All imports loaded successfully.")

def authenticate_user(identifier: str, password: str, remember_me: bool = False, db_path: str = DB_PATH) -> Tuple[bool, Optional[str], Optional[Dict[str, Any]], str]:
    ident_clean = identifier.strip()
    if not ident_clean or not password:
        return False, None, None, "Username/email and password are required."

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        user = conn.execute(
            "SELECT id, username, email, password_hash, role, status, token_version FROM users WHERE username = ? OR email = ?",
            (ident_clean, ident_clean.lower())
        ).fetchone()

        if not user or not verify_password(password, user["password_hash"]):
            return False, None, None, "Invalid username/email or password."
        if user["status"] != "active":
            return False, None, None, "Account is deactivated. Contact administrator."

        hours = 24 if remember_me else 1
        token = create_jwt_token(
            user_id=user["id"],
            username=user["username"],
            role=user["role"],
            token_version=user["token_version"],
            hours=hours
        )
        user_info = {
            "id": user["id"],
            "username": user["username"],
            "email": user["email"],
            "role": user["role"]
        }
        return True, token, user_info, "Authentication successful."
    finally:
        conn.close()

print("Login authentication workflow defined.")


All imports loaded successfully.
Login authentication workflow defined.


In [46]:
def get_user_security_question(identifier: str, db_path: str = DB_PATH) -> Tuple[bool, Optional[str], Optional[str], str]:
    ident_clean = identifier.strip()
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        user = conn.execute(
            "SELECT question_id FROM users WHERE (username = ? OR email = ?) AND status = 'active'",
            (ident_clean, ident_clean.lower())
        ).fetchone()
        if not user:
            return False, None, None, "No active account found."
        q_id = user["question_id"]
        q_text = SECURITY_QUESTIONS.get(q_id, "Security question")
        return True, q_id, q_text, "Question retrieved."
    finally:
        conn.close()

def reset_password_via_security_question(identifier: str, answer: str, new_password: str, confirm_pw: str, db_path: str = DB_PATH) -> Tuple[bool, str]:
    ident_clean = identifier.strip()
    p_err = validate_password_strength(new_password)
    if p_err: return False, p_err
    if new_password != confirm_pw: return False, "Passwords do not match."

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        user = conn.execute(
            "SELECT id, answer_hash FROM users WHERE (username = ? OR email = ?) AND status = 'active'",
            (ident_clean, ident_clean.lower())
        ).fetchone()
        if not user:
            return False, "Account not found."

        if not verify_security_answer(answer, user["answer_hash"]):
            return False, "Incorrect security answer."

        new_h = hash_password(new_password)
        with conn:
            conn.execute("""
                UPDATE users
                SET password_hash = ?, token_version = token_version + 1
                WHERE id = ?
            """, (new_h, user["id"]))
        return True, "Password updated successfully. Prior sessions revoked."
    finally:
        conn.close()

print("Security question password recovery workflow defined.")


Security question password recovery workflow defined.


In [47]:
def request_otp_flow(email: str, db_path: str = DB_PATH) -> Tuple[bool, str]:
    norm_e = email.strip().lower()
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        user = conn.execute("SELECT id, status FROM users WHERE email = ?", (norm_e,)).fetchone()
        if not user or user["status"] != "active":
            return True, "If this email is registered, an OTP has been dispatched."

        # Cooldown check (60 seconds)
        latest = conn.execute("SELECT created_at FROM password_reset_otps WHERE email = ? AND consumed = 0 ORDER BY id DESC LIMIT 1", (norm_e,)).fetchone()
        now = datetime.now(timezone.utc)
        if latest:
            elapsed = (now - datetime.fromisoformat(latest["created_at"])).total_seconds()
            if elapsed < 60:
                return False, f"Please wait {int(60 - elapsed)}s before requesting a new OTP."

        # Invalidate older unconsumed OTPs
        with conn:
            conn.execute("UPDATE password_reset_otps SET consumed = 1 WHERE email = ? AND consumed = 0", (norm_e,))
            raw_otp = f"{secrets.randbelow(900000) + 100000}"
            otp_h = hash_otp(raw_otp)
            exp_iso = (now + timedelta(minutes=5)).isoformat()
            conn.execute("""
                INSERT INTO password_reset_otps (email, otp_hash, created_at, expires_at, attempts, consumed, purpose)
                VALUES (?, ?, ?, ?, 0, 0, 'password_reset')
            """, (norm_e, otp_h, now.isoformat(), exp_iso))

        ok, msg = send_otp_email(norm_e, raw_otp)
        return ok, msg
    finally:
        conn.close()

def reset_password_via_otp(email: str, entered_otp: str, new_password: str, confirm_pw: str, db_path: str = DB_PATH) -> Tuple[bool, str]:
    norm_e = email.strip().lower()
    p_err = validate_password_strength(new_password)
    if p_err: return False, p_err
    if new_password != confirm_pw: return False, "Passwords do not match."

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        record = conn.execute("""
            SELECT id, otp_hash, expires_at, attempts, consumed
            FROM password_reset_otps WHERE email = ? AND consumed = 0
            ORDER BY id DESC LIMIT 1
        """, (norm_e,)).fetchone()

        if not record: return False, "No active OTP found. Please request a new one."

        now = datetime.now(timezone.utc)
        if now > datetime.fromisoformat(record["expires_at"]):
            with conn: conn.execute("UPDATE password_reset_otps SET consumed = 1 WHERE id = ?", (record["id"],))
            return False, "OTP has expired."

        if record["attempts"] >= 5:
            with conn: conn.execute("UPDATE password_reset_otps SET consumed = 1 WHERE id = ?", (record["id"],))
            return False, "Maximum verification attempts exceeded."

        with conn: conn.execute("UPDATE password_reset_otps SET attempts = attempts + 1 WHERE id = ?", (record["id"],))

        computed = hash_otp(entered_otp.strip())
        if not hmac.compare_digest(computed, record["otp_hash"]):
            return False, f"Incorrect OTP. {4 - record['attempts']} attempt(s) remaining."

        # Success: consume OTP and update password
        new_h = hash_password(new_password)
        with conn:
            conn.execute("UPDATE password_reset_otps SET consumed = 1 WHERE id = ?", (record["id"],))
            conn.execute("""
                UPDATE users
                SET password_hash = ?, token_version = token_version + 1
                WHERE email = ?
            """, (new_h, norm_e))
        return True, "Password reset successfully. Prior sessions revoked."
    finally:
        conn.close()

print("Email OTP password recovery workflow defined.")


Email OTP password recovery workflow defined.


In [48]:
%%writefile /content/franchiseops_ai/app.py

import os
import re
import hmac
import time
import secrets
import hashlib
import sqlite3
import smtplib
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage
from typing import Optional, Tuple, Dict, Any

import bcrypt
import jwt
import streamlit as st

# ==============================================================================
# 1. BRAND CONSTANTS & DESIGN SYSTEM
# ==============================================================================
COLOR_EMERALD_PRIMARY = "#176B45"
COLOR_DEEP_FOREST = "#0B3D2E"
COLOR_BRIGHT_ACCENT = "#18A86B"
COLOR_WARM_IVORY = "#F7F5EF"
COLOR_WHITE = "#FFFFFF"
COLOR_DARK_TEXT = "#17231D"
COLOR_MUTED_TEXT = "#68746D"
COLOR_LIGHT_BORDER = "#DCE5DE"
COLOR_ALERT_BG = "#EBF5F0"
COLOR_DANGER = "#D9383A"
COLOR_WARNING = "#E59500"

SECURITY_QUESTIONS = {
    "Q1": "What was the name of your first school?",
    "Q2": "What was the name of your first pet?",
    "Q3": "What city were you born in?",
    "Q4": "What was your childhood nickname?"
}

# ==============================================================================
# 2. CONFIGURATION & SECRETS MANAGEMENT
# ==============================================================================
def get_secret(key: str, default: Optional[str] = None) -> Optional[str]:
    """
    Safely retrieve secrets from Google Colab userdata if available,
    falling back to os.environ or a default value.
    Never prints or leaks secret values.
    """
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(key)
        if val is not None and str(val).strip():
            return str(val).strip()
    except Exception:
        pass

    env_val = os.environ.get(key)
    if env_val is not None and str(env_val).strip():
        return str(env_val).strip()

    return default

# Read secrets
@st.cache_resource
def _ephemeral_jwt_secret() -> str:
    return secrets.token_hex(32)  # random per server run; never stored in code

JWT_SECRET = get_secret("JWT_SECRET") or _ephemeral_jwt_secret()
EMAIL_ADDRESS = get_secret("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = get_secret("EMAIL_PASSWORD", "")
ADMIN_USERNAME = get_secret("ADMIN_USERNAME", "admin")
ADMIN_PASSWORD = get_secret("ADMIN_PASSWORD", "")

# Database Path Resolution (supports Colab /content and local dev)
def resolve_db_path() -> str:
    custom_dir = os.environ.get("FRANCHISEOPS_DB_DIR")
    if custom_dir:
        os.makedirs(custom_dir, exist_ok=True)
        return os.path.join(custom_dir, "users.db")
    colab_default_dir = "/content/franchiseops_ai/data"
    if os.path.exists("/content") or os.access("/content", os.W_OK):
        os.makedirs(colab_default_dir, exist_ok=True)
        return os.path.join(colab_default_dir, "users.db")
    else:
        local_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")
        os.makedirs(local_dir, exist_ok=True)
        return os.path.join(local_dir, "users.db")

DB_PATH = resolve_db_path()

# ==============================================================================
# 3. DATABASE ENGINE & SCHEMAS
# ==============================================================================
def get_db_connection() -> sqlite3.Connection:
    """Returns a SQLite connection with Row factory enabled."""
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn

def init_db() -> None:
    """
    Idempotent database initialization.
    Creates users and password_reset_otps tables safely without wiping data.
    """
    conn = get_db_connection()
    try:
        with conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS users (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    username TEXT UNIQUE NOT NULL,
                    email TEXT UNIQUE NOT NULL,
                    password_hash TEXT NOT NULL,
                    question_id TEXT NOT NULL,
                    answer_hash TEXT NOT NULL,
                    role TEXT NOT NULL DEFAULT 'user',
                    created_at TEXT NOT NULL,
                    status TEXT NOT NULL DEFAULT 'active',
                    token_version INTEGER NOT NULL DEFAULT 1
                )
            """)
            conn.execute("""
                CREATE TABLE IF NOT EXISTS password_reset_otps (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    email TEXT NOT NULL,
                    otp_hash TEXT NOT NULL,
                    created_at TEXT NOT NULL,
                    expires_at TEXT NOT NULL,
                    attempts INTEGER NOT NULL DEFAULT 0,
                    consumed INTEGER NOT NULL DEFAULT 0,
                    purpose TEXT NOT NULL DEFAULT 'password_reset'
                )
            """)
            conn.execute("CREATE INDEX IF NOT EXISTS idx_users_username ON users(username)")
            conn.execute("CREATE INDEX IF NOT EXISTS idx_users_email ON users(email)")
            conn.execute("CREATE INDEX IF NOT EXISTS idx_otps_email ON password_reset_otps(email)")
    finally:
        conn.close()

# Initialize DB on load
init_db()

# ==============================================================================
# 4. VALIDATION & CRYPTOGRAPHIC UTILITIES
# ==============================================================================
EMAIL_REGEX = re.compile(r"^[a-zA-Z0-9._%+-]{2,}@[a-zA-Z0-9.-]{2,}\.[a-zA-Z]{2,}$")
USERNAME_REGEX = re.compile(r"^[a-zA-Z0-9_]{3,30}$")

def validate_username(username: str) -> Optional[str]:
    u = username.strip()
    if not u:
        return "Username cannot be empty."
    if not USERNAME_REGEX.match(u):
        return "Username must be 3-30 characters and contain only letters, numbers, and underscores."
    return None

def validate_email(email: str) -> Optional[str]:
    e = email.strip()
    if not e:
        return "Email cannot be empty."
    if not EMAIL_REGEX.match(e):
        return "Enter a valid email (at least 2 chars before @, valid domain, and 2+ char extension, e.g. user@franchise.com)."
    return None

def validate_password_strength(password: str) -> Optional[str]:
    if len(password) < 8:
        return "Password must be at least 8 characters long."
    if not re.search(r"[A-Z]", password):
        return "Password must contain at least one uppercase letter (A-Z)."
    if not re.search(r"[a-z]", password):
        return "Password must contain at least one lowercase letter (a-z)."
    if not re.search(r"\d", password):
        return "Password must contain at least one digit (0-9)."
    if not re.search(r"[!@#$%^&*(),.?\":{}|<>\-_=+]", password):
        return "Password must contain at least one special character (!@#$%^&*...)."
    return None

def hash_password(plaintext: str) -> str:
    """Hash password using bcrypt with random salt."""
    salt = bcrypt.gensalt(rounds=12)
    return bcrypt.hashpw(plaintext.encode("utf-8"), salt).decode("utf-8")

def verify_password(plaintext: str, hashed: str) -> bool:
    """Verify password against bcrypt hash."""
    try:
        return bcrypt.checkpw(plaintext.encode("utf-8"), hashed.encode("utf-8"))
    except Exception:
        return False

def hash_security_answer(answer: str) -> str:
    """Normalize and hash security answer using SHA-256 with project salt."""
    normalized = answer.strip().lower()
    salt = "FranchiseOpsAI_SecAnsSalt_2026"
    return hashlib.sha256(f"{salt}_{normalized}".encode("utf-8")).hexdigest()

def verify_security_answer(answer: str, stored_hash: str) -> bool:
    """Constant-time verification of security answer."""
    computed = hash_security_answer(answer)
    return hmac.compare_digest(computed, stored_hash)

def hash_otp(otp: str) -> str:
    """Hash OTP using SHA-256 with salt."""
    salt = "FranchiseOpsAI_OTPSalt_2026"
    return hashlib.sha256(f"{salt}_{otp}".encode("utf-8")).hexdigest()

# ==============================================================================
# 5. JWT SESSION MANAGEMENT & REVOCATION
# ==============================================================================
def create_jwt_token(user_id: int, username: str, role: str, token_version: int, hours: int = 1) -> str:
    """
    Creates a signed JWT with HS256 algorithm and standard claims:
    sub (user id), username, role, ver (token version), iat, exp, jti.
    """
    now = datetime.now(timezone.utc)
    payload = {
        "sub": str(user_id),
        "username": username,
        "role": role,
        "ver": token_version,
        "iat": int(now.timestamp()),
        "exp": int((now + timedelta(hours=hours)).timestamp()),
        "jti": secrets.token_hex(16)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_jwt_token(token: str) -> Tuple[bool, Optional[Dict[str, Any]], str]:
    """
    Verifies token signature, expiry, and queries the database to ensure
    the user exists, is active, and the token version matches (session revocation check).
    """
    if not token:
        return False, None, "No token provided."
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except jwt.ExpiredSignatureError:
        return False, None, "Session expired. Please log in again."
    except jwt.InvalidTokenError:
        return False, None, "Invalid authentication token."

    # Server-side validation against database
    user_id = payload.get("sub")
    token_ver = payload.get("ver")

    # Handle dedicated server-side admin token
    if user_id == "admin_0" and payload.get("role") == "admin":
        return True, payload, "Admin session valid."

    conn = get_db_connection()
    try:
        user = conn.execute(
            "SELECT id, username, role, status, token_version FROM users WHERE id = ?",
            (user_id,)
        ).fetchone()
        if not user:
            return False, None, "User account no longer exists."
        if user["status"] != "active":
            return False, None, "User account has been deactivated."
        if user["token_version"] != token_ver:
            return False, None, "Session has been invalidated due to a password reset."

        return True, {
            "sub": user["id"],
            "username": user["username"],
            "role": user["role"],
            "status": user["status"]
        }, "Valid session."
    finally:
        conn.close()

# ==============================================================================
# 6. GMAIL SMTP SERVICE & OTP RECOVERY ENGINE
# ==============================================================================
def send_otp_email(to_email: str, otp_code: str) -> Tuple[bool, str]:
    """
    Sends a 6-digit OTP email using Gmail SMTP and standard EmailMessage.
    Requires EMAIL_ADDRESS and EMAIL_PASSWORD (App Password).
    """
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        return False, "Gmail credentials not configured. Please set EMAIL_ADDRESS and EMAIL_PASSWORD in Colab Secrets."

    msg = EmailMessage()
    msg["Subject"] = f"FranchiseOps AI - Password Reset OTP: {otp_code}"
    msg["From"] = f"FranchiseOps AI Security <{EMAIL_ADDRESS}>"
    msg["To"] = to_email

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
      <meta charset="utf-8">
      <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; background: #F7F5EF; margin: 0; padding: 24px; color: #17231D; }}
        .card {{ max-width: 480px; margin: 0 auto; background: #ffffff; border-radius: 12px; border: 1px solid #DCE5DE; padding: 32px; box-shadow: 0 4px 12px rgba(11,61,46,0.06); }}
        .header {{ text-align: center; border-bottom: 2px solid #176B45; padding-bottom: 16px; margin-bottom: 24px; }}
        .title {{ color: #176B45; font-size: 22px; font-weight: bold; margin: 0; }}
        .subtitle {{ color: #68746D; font-size: 13px; margin-top: 4px; }}
        .otp-box {{ background: #EBF5F0; border: 1px dashed #18A86B; border-radius: 8px; text-align: center; padding: 18px; margin: 24px 0; }}
        .otp-code {{ font-size: 32px; font-weight: 800; letter-spacing: 6px; color: #0B3D2E; font-family: monospace; }}
        .warning {{ font-size: 12px; color: #68746D; line-height: 1.5; margin-top: 16px; border-left: 3px solid #176B45; padding-left: 10px; }}
        .footer {{ text-align: center; font-size: 11px; color: #68746D; margin-top: 24px; }}
      </style>
    </head>
    <body>
      <div class="card">
        <div class="header">
          <h1 class="title">FranchiseOps AI</h1>
          <div class="subtitle">Agentic Franchise Management System — Security Service</div>
        </div>
        <p>Hello,</p>
        <p>You requested a one-time password (OTP) to reset your FranchiseOps AI password. Use the verification code below:</p>
        <div class="otp-box">
          <div class="otp-code">{otp_code}</div>
        </div>
        <div class="warning">
          <strong>Security Notice:</strong>
          <ul>
            <li>This OTP is strictly valid for <strong>5 minutes</strong>.</li>
            <li>It can only be used once and allows up to 5 verification attempts.</li>
            <li>Never share this code with anyone. FranchiseOps AI staff will never request your OTP.</li>
          </ul>
        </div>
        <p>If you did not initiate this request, you can safely disregard this email; your password remains unchanged.</p>
        <div class="footer">
          Infosys Springboard Internship 7.0 &bull; Batch 4 &bull; FranchiseOps AI Module
        </div>
      </div>
    </body>
    </html>
    """
    msg.set_content(f"Your FranchiseOps AI verification code is: {otp_code}. Valid for 5 minutes. Do not share.")
    msg.add_alternative(html_content, subtype="html")

    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=15) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
            server.send_message(msg)
        return True, "OTP successfully delivered to your registered email address."
    except Exception as e:
        return False, f"Email delivery failed. Please verify your Gmail App Password configuration. ({type(e).__name__})"

def request_password_reset_otp(email: str) -> Tuple[bool, str]:
    """
    Handles OTP generation, cooldown check, database storage, and email delivery.
    """
    norm_email = email.strip().lower()
    conn = get_db_connection()
    try:
        user = conn.execute("SELECT id, status FROM users WHERE email = ?", (norm_email,)).fetchone()
        if not user or user["status"] != "active":
            return False, "No active account is registered with this email address. Please sign up first."

        # Check 60-second cooldown on latest unconsumed OTP
        latest_otp = conn.execute("""
            SELECT created_at FROM password_reset_otps
            WHERE email = ? AND consumed = 0
            ORDER BY id DESC LIMIT 1
        """, (norm_email,)).fetchone()

        now = datetime.now(timezone.utc)
        if latest_otp:
            last_created = datetime.fromisoformat(latest_otp["created_at"])
            elapsed = (now - last_created).total_seconds()
            if elapsed < 60:
                return False, f"Please wait {int(60 - elapsed)} seconds before requesting a new OTP."

        # Invalidate previous unconsumed OTPs for this email
        with conn:
            conn.execute("UPDATE password_reset_otps SET consumed = 1 WHERE email = ? AND consumed = 0", (norm_email,))

            # Generate cryptographically secure 6-digit OTP
            raw_otp = f"{secrets.randbelow(900000) + 100000}"
            otp_h = hash_otp(raw_otp)
            expires_at = (now + timedelta(minutes=5)).isoformat()

            conn.execute("""
                INSERT INTO password_reset_otps (email, otp_hash, created_at, expires_at, attempts, consumed, purpose)
                VALUES (?, ?, ?, ?, 0, 0, 'password_reset')
            """, (norm_email, otp_h, now.isoformat(), expires_at))

        # Dispatch email
        ok, msg = send_otp_email(norm_email, raw_otp)
        if not ok:
            return False, msg
        return True, "A 6-digit OTP has been sent to your email. Valid for 5 minutes."
    finally:
        conn.close()

def verify_and_consume_otp(email: str, entered_otp: str) -> Tuple[bool, str]:
    """
    Verifies submitted OTP against database records with constant-time matching,
    attempt limit checks (max 5), expiry checks, and one-time use consumption.
    """
    norm_email = email.strip().lower()
    clean_otp = entered_otp.strip()
    if len(clean_otp) != 6 or not clean_otp.isdigit():
        return False, "OTP must be a 6-digit number."

    conn = get_db_connection()
    try:
        record = conn.execute("""
            SELECT id, otp_hash, expires_at, attempts, consumed
            FROM password_reset_otps
            WHERE email = ? AND consumed = 0
            ORDER BY id DESC LIMIT 1
        """, (norm_email,)).fetchone()

        if not record:
            return False, "No active OTP request found. Please request a new OTP."

        now = datetime.now(timezone.utc)
        expires_at = datetime.fromisoformat(record["expires_at"])

        if now > expires_at:
            with conn:
                conn.execute("UPDATE password_reset_otps SET consumed = 1 WHERE id = ?", (record["id"],))
            return False, "This OTP has expired. Please request a new one."

        if record["attempts"] >= 5:
            with conn:
                conn.execute("UPDATE password_reset_otps SET consumed = 1 WHERE id = ?", (record["id"],))
            return False, "Maximum verification attempts exceeded. This OTP has been invalidated."

        # Increment attempt counter
        with conn:
            conn.execute("UPDATE password_reset_otps SET attempts = attempts + 1 WHERE id = ?", (record["id"],))

        computed_hash = hash_otp(clean_otp)
        if not hmac.compare_digest(computed_hash, record["otp_hash"]):
            remaining = 4 - record["attempts"]
            return False, f"Incorrect OTP. {max(0, remaining)} attempt(s) remaining."

        # Valid: mark as consumed
        with conn:
            conn.execute("UPDATE password_reset_otps SET consumed = 1 WHERE id = ?", (record["id"],))
        return True, "OTP verified successfully. You may now reset your password."
    finally:
        conn.close()

# Navigation and session state defaults
def init_session_state() -> None:
    """Initialize session state defaults safely inside Streamlit execution context."""
    if "page" not in st.session_state:
        st.session_state.page = "Login"
    if "jwt_token" not in st.session_state:
        st.session_state.jwt_token = None
    if "current_user" not in st.session_state:
        st.session_state.current_user = None
    if "reset_authorized_email" not in st.session_state:
        st.session_state.reset_authorized_email = None
    if "security_q_user" not in st.session_state:
        st.session_state.security_q_user = None

def navigate_to(page_name: str) -> None:
    st.session_state.page = page_name
    st.rerun()

def apply_custom_styles() -> None:
    """Apply the Emerald Green SaaS theme styles."""
    st.markdown(f"""
<style>
/* Global resets & typography */
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap');
html, body, [class*="css"] {{
    font-family: 'Plus Jakarta Sans', -apple-system, BlinkMacSystemFont, sans-serif;
    color: {COLOR_DARK_TEXT};
}}

.stApp {{
    background-color: {COLOR_WARM_IVORY};
}}

/* Main container card styling */
.block-container {{
    max-width: 540px;
    margin-top: 2rem;
    padding: 2.2rem 2.4rem !important;
    background: {COLOR_WHITE};
    border: 1px solid {COLOR_LIGHT_BORDER};
    border-radius: 16px;
    box-shadow: 0 10px 30px rgba(11, 61, 46, 0.08);
}}

/* Brand banner header */
.brand-header {{
    text-align: center;
    margin-bottom: 1.5rem;
}}
.brand-title {{
    color: {COLOR_EMERALD_PRIMARY};
    font-size: 1.75rem;
    font-weight: 800;
    letter-spacing: -0.5px;
    margin: 0;
}}
.brand-subtitle {{
    color: {COLOR_MUTED_TEXT};
    font-size: 0.85rem;
    font-weight: 500;
    margin-top: 0.25rem;
}}
.brand-tagline {{
    display: inline-block;
    background: {COLOR_ALERT_BG};
    color: {COLOR_DEEP_FOREST};
    padding: 0.25rem 0.75rem;
    border-radius: 999px;
    font-size: 0.75rem;
    font-weight: 700;
    margin-top: 0.5rem;
    border: 1px solid {COLOR_LIGHT_BORDER};
}}

/* Form elements */
div[data-testid="stForm"] {{
    border: none;
    padding: 0;
}}
label p, div[data-testid="stWidgetLabel"] p {{
    color: {COLOR_DARK_TEXT} !important;
    font-weight: 600 !important;
    font-size: 0.88rem !important;
    margin-bottom: 0.2rem !important;
}}
div[data-baseweb="input"], div[data-baseweb="select"] > div {{
    background: #FAFAFA !important;
    border: 1.5px solid {COLOR_LIGHT_BORDER} !important;
    border-radius: 8px !important;
}}
div[data-baseweb="input"]:focus-within {{
    border-color: {COLOR_BRIGHT_ACCENT} !important;
    box-shadow: 0 0 0 3px rgba(24, 168, 107, 0.15) !important;
}}

/* Buttons */
button[data-testid^="stBaseButton-primary"] {{
    background: linear-gradient(135deg, {COLOR_EMERALD_PRIMARY} 0%, {COLOR_DEEP_FOREST} 100%) !important;
    border: none !important;
    border-radius: 8px !important;
    color: #ffffff !important;
    font-weight: 600 !important;
    font-size: 0.95rem !important;
    height: 2.8rem !important;
    box-shadow: 0 4px 12px rgba(11, 61, 46, 0.2) !important;
    transition: all 0.2s ease !important;
}}
button[data-testid^="stBaseButton-primary"]:hover {{
    filter: brightness(1.1) !important;
    transform: translateY(-1px) !important;
}}
button[data-testid^="stBaseButton-secondary"] {{
    background: #FFFFFF !important;
    border: 1.5px solid {COLOR_LIGHT_BORDER} !important;
    border-radius: 8px !important;
    color: {COLOR_DEEP_FOREST} !important;
    font-weight: 600 !important;
    font-size: 0.88rem !important;
    height: 2.5rem !important;
}}
button[data-testid^="stBaseButton-secondary"]:hover {{
    background: {COLOR_ALERT_BG} !important;
    border-color: {COLOR_EMERALD_PRIMARY} !important;
}}

/* Status badges */
.status-badge-active {{
    background: #D4EDDA;
    color: #155724;
    padding: 0.25rem 0.6rem;
    border-radius: 6px;
    font-size: 0.8rem;
    font-weight: 700;
}}
.module-card {{
    background: #FAFAFA;
    border: 1px solid {COLOR_LIGHT_BORDER};
    border-radius: 10px;
    padding: 1rem;
    margin-bottom: 0.8rem;
}}
.module-card h4 {{
    margin: 0 0 0.3rem 0;
    color: {COLOR_DEEP_FOREST};
    font-size: 0.95rem;
}}
.module-card p {{
    margin: 0;
    font-size: 0.82rem;
    color: {COLOR_MUTED_TEXT};
}}

/* forms as clean cards, bolder tabs */
[data-testid="stForm"] {{
    border: 1px solid {COLOR_LIGHT_BORDER} !important;
    border-radius: 12px !important;
    padding: 1.2rem 1.2rem 0.8rem 1.2rem !important;
    background: #FFFFFF;
}}
.stTabs [data-baseweb="tab"] p {{ font-weight: 600; }}
.otp-card {{
    background: {COLOR_ALERT_BG};
    border: 1px solid {COLOR_LIGHT_BORDER};
    border-radius: 10px;
    padding: 0.8rem 1rem;
    margin: 1rem 0 0.8rem 0;
    font-size: 0.88rem;
    color: {COLOR_DEEP_FOREST};
}}
.otp-card b {{ word-break: break-all; }}
.step-title {{
    font-size: 0.8rem; font-weight: 700; letter-spacing: 0.06em;
    text-transform: uppercase; color: {COLOR_MUTED_TEXT}; margin: 0.4rem 0 0.4rem 0;
}}
/* hide Streamlit's "Press Enter to submit form" hint */
[data-testid="InputInstructions"] {{
    display: none !important;
}}
</style>
""", unsafe_allow_html=True)


# ==============================================================================
# 7b. PROFESSIONAL UI KIT (logo, hero, stepper, cards, styles)
# ==============================================================================
LEAF_SVG = ('<svg width="40" height="40" viewBox="0 0 48 48"><path d="M24 44C10 40 6 26 8 10c14 0 28 6 26 22-1 6-5 10-10 12z" fill="#176B45"/>'
            '<path d="M26 40c0-12 6-20 16-26-2 14-6 24-16 26z" fill="#18A86B"/></svg>')

LEAF_LIGHT = ('<svg width="40" height="40" viewBox="0 0 48 48"><path d="M24 44C10 40 6 26 8 10c14 0 28 6 26 22-1 6-5 10-10 12z" fill="#18A86B"/>'
              '<path d="M26 40c0-12 6-20 16-26-2 14-6 24-16 26z" fill="#8FF0C3"/></svg>')

def _h(html: str) -> str:
    """Collapse HTML to one line so Markdown never treats it as a code block."""
    return "".join(line.strip() for line in html.strip().splitlines())

PRO_CSS = """
<style>
#MainMenu, footer {visibility: hidden;}
[data-testid="stToolbar"], [data-testid="stDecoration"] {display: none;}
header[data-testid="stHeader"] {background: transparent;}
.stApp {background: radial-gradient(circle at 0% 0%, #DCEFE5 0, transparent 26%), radial-gradient(circle at 100% 100%, #DCEFE5 0, transparent 30%), #F6FAF8;}
.block-container {max-width: 600px !important; margin-top: 0; padding: 1.5rem 1rem 1rem !important; background: transparent !important; border: none !important; box-shadow: none !important; position: relative; z-index: 1;}
.st-key-authcard {background:#fff; border:1px solid #E3ECE6; border-radius:18px; padding:2rem 2.2rem; box-shadow:0 12px 40px rgba(11,61,46,.10);}
.stApp::before {content:""; position:fixed; top:-130px; left:-130px; width:340px; height:340px; border-radius:50%; background:#DDEFE6; z-index:0;}
.stApp::after {content:""; position:fixed; bottom:-150px; right:-150px; width:400px; height:400px; border-radius:50%; background:#DDEFE6; z-index:0;}
.logo-row {display:flex; justify-content:center; align-items:center; gap:.6rem;}
.logo-text {font-size:1.9rem; font-weight:800; color:#0B3D2E; letter-spacing:-.5px;}
.logo-text span {color:#18A86B;}
.tag {text-align:center; color:#68746D; font-size:.85rem; margin:.1rem 0 1.1rem;}
.hero-icon {width:64px; height:64px; border-radius:50%; background:#E3F2EA; display:flex; align-items:center; justify-content:center; font-size:1.7rem; margin:0 auto .6rem;}
.hero-title {text-align:center; font-size:1.5rem; font-weight:800; color:#0B3D2E; margin:0;}
.hero-sub {text-align:center; color:#68746D; font-size:.9rem; margin:.3rem 0 1rem;}
.stepper {display:flex; gap:1rem; border-bottom:2px solid #E5ECE8; margin:.8rem 0 1.2rem;}
.step {flex:1; display:flex; align-items:center; gap:.5rem; padding-bottom:.6rem; color:#9AA7A0; font-weight:600; font-size:.9rem; border-bottom:2px solid transparent; margin-bottom:-2px;}
.step b {width:26px; height:26px; border-radius:50%; background:#C9D2CD; color:#fff; display:flex; align-items:center; justify-content:center; font-size:.8rem;}
.step.on {color:#176B45; border-bottom-color:#176B45;}
.step.on b {background:#176B45;}
.secure {background:#EEF7F2; border-radius:10px; padding:.6rem; text-align:center; color:#176B45; font-size:.82rem; margin:.8rem 0;}
.foot {text-align:center; color:#8A968F; font-size:.78rem; margin-top:1.2rem;}
div[data-baseweb="input"], div[data-baseweb="select"] > div {background:#fff !important; border:1px solid #D5DED8 !important; border-radius:12px !important;}
.stButton > button, [data-testid="stFormSubmitButton"] > button {border-radius:12px; font-weight:700; padding:.65rem 1rem;}
button[kind="primary"], button[kind="primaryFormSubmit"] {background:linear-gradient(90deg,#176B45,#1E8A5A) !important; border:none !important; color:#fff !important;}
[data-testid="InputInstructions"] {display:none !important;}
div[data-baseweb="base-input"], div[data-baseweb="input"] input {background:#fff !important;}
div[data-baseweb="input"]:focus-within {border-color:#176B45 !important; box-shadow:0 0 0 3px rgba(23,107,69,.15) !important;}
.stButton > button[kind="secondary"] {background:#fff; border:1px solid #CFE0D6; color:#176B45;}
.stButton > button[kind="secondary"]:hover {background:#EEF7F2; border-color:#176B45; color:#176B45;}
[data-baseweb="tab-highlight"] {background:#176B45 !important;}
</style>
"""

DASH_CSS = """
<style>
.block-container {max-width:1280px !important; background:transparent !important; border:none !important; box-shadow:none !important; padding:1rem 1.5rem !important; margin-top:0 !important;}
.topbar {display:flex; justify-content:space-between; align-items:center; background:#fff; border:1px solid #DCE5DE; border-radius:14px; padding:.7rem 1.1rem; margin-bottom:1.2rem;}
.nav {display:flex; gap:.5rem; align-items:center;}
.pill {padding:.35rem .9rem; border-radius:10px; font-weight:600; font-size:.88rem; color:#68746D;}
.pill.on {background:#E3F2EA; color:#176B45;}
.chip {display:flex; align-items:center; gap:.6rem; font-weight:700; color:#0B3D2E;}
.avatar {width:38px; height:38px; border-radius:50%; background:#176B45; color:#fff; display:flex; align-items:center; justify-content:center; font-weight:800;}
.avatar.lg {width:56px; height:56px; font-size:1.2rem;}
.welcome h1 {font-size:1.8rem; font-weight:800; color:#0B3D2E; margin:0;}
.welcome p {color:#68746D; margin:.2rem 0 1.2rem;}
.cards, .mods {display:grid; grid-template-columns:repeat(4,1fr); gap:1rem; margin-bottom:1.2rem;}
.card, .stat {background:#fff; border:1px solid #DCE5DE; border-radius:14px; padding:1rem 1.1rem; box-shadow:0 2px 10px rgba(11,61,46,.04);}
.card h4 {margin:.2rem 0; color:#0B3D2E; font-size:.98rem;}
.card p {margin:0; color:#68746D; font-size:.8rem;}
.stat .ic {width:40px; height:40px; border-radius:10px; display:flex; align-items:center; justify-content:center; font-size:1.2rem; margin-bottom:.6rem;}
.stat .lb {color:#68746D; font-size:.85rem;}
.stat .vl {font-size:1.9rem; font-weight:800; color:#0B3D2E;}
.stat .dl.up {color:#18A86B;}
.stat .dl {font-size:.78rem; color:#18A86B; font-weight:600;}
.sec-title {font-weight:800; color:#0B3D2E; margin:0 0 .5rem; font-size:1.02rem;}
.qa {display:flex; justify-content:space-between; align-items:center; border:1px solid #DCE5DE; border-radius:10px; padding:.65rem .8rem; margin-top:.6rem; font-weight:600; color:#0B3D2E; font-size:.9rem;}
.copilot {background:linear-gradient(90deg,#E7F5EE,#F3FAF6); border:1px solid #CFE6DA; border-radius:14px; padding:1.1rem 1.3rem; margin:1.2rem 0;}
.copilot h3 {margin:0 0 .2rem; color:#0B3D2E;}
.copilot p {margin:0; color:#68746D; font-size:.88rem;}
.act {display:flex; gap:.7rem; padding:.55rem 0; border-bottom:1px solid #EEF2EF; font-size:.85rem; color:#17231D;}
.act small {display:block; color:#68746D;}
.badge {display:inline-block; background:#DDF3E8; color:#176B45; border-radius:999px; padding:.1rem .6rem; font-size:.75rem; font-weight:700;}
.dot {display:inline-block; width:9px; height:9px; border-radius:50%; background:#18A86B; margin-right:.4rem;}
.side-foot {color:#68746D; font-size:.8rem; margin:2rem 0 1rem;}
.row {display:flex; justify-content:space-between; padding:.7rem 0; border-bottom:1px solid #EEF2EF; font-size:.92rem;}
.row span:first-child {color:#68746D;}
.st-key-navpanel {background:linear-gradient(180deg,#0B3D2E 0%,#082A20 100%); border:none; border-radius:16px; padding:1.1rem .9rem; min-height:88vh;}
.st-key-navpanel .logo-text {color:#fff !important;} .st-key-navpanel .logo-text span {color:#8FF0C3 !important;}
.st-key-navpanel .side-status {color:#BFD8CB; font-size:.8rem;}
.st-key-navpanel .side-tag {color:#8FF0C3; font-family:'Segoe Script','Brush Script MT',cursive; font-size:1.05rem; line-height:1.5; margin:1rem 0;}
.hero {display:flex; align-items:center; gap:1.2rem; border-radius:18px; padding:1.6rem 1.8rem; color:#fff; margin-bottom:1rem;}
.hero.sun {background:linear-gradient(110deg,#FBE9B7 0%,#BFE3CF 45%,#5FA989 100%); color:#0B3D2E;}
.hero.night {background:linear-gradient(110deg,#1B2B6B 0%,#3B2A7A 55%,#0B3D2E 100%);}
.hero h2 {margin:0; font-size:1.5rem; font-weight:800; color:inherit;}
.hero p {margin:.2rem 0 0; opacity:.85; font-size:.9rem;}
.avatar.xl {width:72px; height:72px; font-size:1.6rem; border:3px solid #fff;}
.pill-ok {background:#DDF3E8; color:#176B45; border-radius:999px; padding:.15rem .7rem; font-size:.78rem; font-weight:700;}
.stTabs [data-baseweb="tab-list"] {gap:.4rem; background:#EEF7F2; border-radius:12px; padding:.3rem; border-bottom:none;}
.stTabs [data-baseweb="tab"] {border-radius:10px; padding:.45rem 1rem; height:auto;}
.stTabs [aria-selected="true"] {background:#fff; box-shadow:0 1px 4px rgba(11,61,46,.12);}
.stTabs [data-baseweb="tab-highlight"], .stTabs [data-baseweb="tab-border"] {display:none;}
.stat .dl.down {color:#E5484D;}
.st-key-growth_panel {position:relative;}
.st-key-growth_panel [data-testid="stSelectbox"] {position:absolute; top:.85rem; right:1.1rem; width:165px; z-index:3;}
.st-key-nav_bottom {margin-top:auto;}
.st-key-qa_panel button > div {width:auto !important; flex:1;}
.st-key-qa_panel button::after {content:"›"; margin-left:auto; font-size:1.3rem; color:#9AA7A0;}
.st-key-navpanel .stButton > button {justify-content:flex-start; text-align:left; width:100%; border-radius:12px; padding:.6rem .9rem;}
.st-key-navpanel button[kind="secondary"] {background:transparent !important; border:1px solid transparent !important; color:#D6EBDD !important; box-shadow:none !important;}
.st-key-navpanel button[kind="secondary"]:hover {background:rgba(255,255,255,.09) !important;}
.st-key-navpanel button[kind="primary"] {background:rgba(255,255,255,.16) !important; color:#fff !important; border:1px solid rgba(255,255,255,.22) !important; box-shadow:none !important;}
[data-testid="stSidebar"], [data-testid="stSidebarCollapsedControl"] {display:none !important;}
.st-key-navpanel button > div, .st-key-qa_panel button > div {justify-content:flex-start !important; width:100%;}
.st-key-navpanel button p, .st-key-qa_panel button p {text-align:left !important; width:100%;}
.st-key-qa_panel, .st-key-growth_panel {background:#fff; border:1px solid #DCE5DE; border-radius:14px; padding:1rem 1.1rem; box-shadow:0 2px 10px rgba(11,61,46,.04);}
.st-key-qa_panel .stButton > button {width:100%; border-radius:10px; padding:.6rem .8rem; font-weight:600;}
.empty {display:flex; flex-direction:column; align-items:center; justify-content:center; text-align:center; color:#68746D; padding:2.4rem 1rem; font-size:.88rem;}
.empty b {color:#0B3D2E; font-size:1.05rem; margin:.4rem 0 .2rem;}
.sbstat {display:flex; gap:.6rem; align-items:center; background:#EEF7F2; border:1px solid #CFE6DA; border-radius:12px; padding:.7rem .8rem; margin:.6rem 0 .8rem; font-size:.85rem; color:#0B3D2E;}
.sbstat small {display:block; color:#68746D; font-size:.72rem;}
.mods3 {display:grid; grid-template-columns:repeat(3,1fr); gap:1rem; margin-bottom:1.2rem;}
.mod {display:flex; gap:.8rem; align-items:center; background:#fff; border:1px solid #DCE5DE; border-radius:14px; padding:1rem; box-shadow:0 2px 10px rgba(11,61,46,.04);}
.mod .mi {width:46px; height:46px; flex:none; border-radius:12px; background:#E3F2EA; display:flex; align-items:center; justify-content:center; font-size:1.4rem;}
.mod h4 {margin:0 0 .15rem; color:#0B3D2E; font-size:.98rem;}
.mod p {margin:0; color:#68746D; font-size:.78rem;}
.pillbtn {margin-left:auto; flex:none; border:1px solid #CFE0D6; color:#176B45; border-radius:999px; padding:.25rem .7rem; font-size:.72rem; font-weight:700; white-space:nowrap;}
.banner {display:flex; justify-content:space-between; align-items:center; gap:1rem; background:linear-gradient(110deg,#0B3D2E,#176B45 70%,#18A86B); border-radius:16px; padding:1.5rem 1.8rem; margin:0 0 1.2rem; color:#fff;}
.banner h2 {margin:0 0 .3rem; font-size:1.5rem; font-weight:800; color:#fff;}
.banner h2 span {color:#8FF0C3;}
.banner p {margin:0; color:#D6F3E5; font-size:.9rem;}
.chips {display:flex; gap:.5rem; flex-wrap:wrap; justify-content:flex-end;}
.chips span {border:1px solid rgba(255,255,255,.4); border-radius:999px; padding:.3rem .8rem; font-size:.78rem; font-weight:600;}
@media (max-width:900px) {.cards, .mods {grid-template-columns:repeat(2,1fr);} .mods3 {grid-template-columns:1fr;} .banner {flex-direction:column; align-items:flex-start;}}
</style>
"""

DARK_CSS = """
<style>
.stApp {background:#0B1512 !important;}
.stApp, .stApp p, .stApp label, .stApp span, .stApp li, .stApp h1, .stApp h2, .stApp h3, .stApp h4, [data-testid="stWidgetLabel"] p {color:#E6F4EC;}
.hero-title, .logo-text, .welcome h1, .sec-title, .card h4, .mod h4, .stat .vl, .chip {color:#E6F4EC !important;}
.tag, .hero-sub, .stat .lb, .card p, .mod p, .welcome p, .foot, .side-foot, .row span:first-child, .sbstat small, .act small {color:#9FB3A9 !important;}
.block-container {background:transparent !important; border:none !important;}
.card, .stat, .mod, .topbar {background:#12201B !important; border-color:#22362E !important;}
.act, .row {border-bottom-color:#22362E !important; color:#E6F4EC;}
.secure, .sbstat {background:#14261F !important; border-color:#22362E !important; color:#8FF0C3 !important;}
.hero-icon, .mod .mi {background:#173B2D !important;}
.badge {background:#173B2D !important; color:#8FF0C3 !important;}
.pillbtn {border-color:#2F4A3F !important; color:#8FF0C3 !important;}
.stepper {border-bottom-color:#22362E !important;}
div[data-baseweb="base-input"], div[data-baseweb="input"] input, div[data-baseweb="select"] > div {background:#0F1A16 !important; color:#E6F4EC !important;}
div[data-baseweb="input"], div[data-baseweb="select"] > div {border-color:#2A4036 !important;}
.stButton > button[kind="secondary"] {background:#12201B !important; border-color:#2A4036 !important; color:#8FF0C3 !important;}
[data-testid="stForm"] {background:#12201B !important; border-color:#22362E !important;}
.st-key-navpanel, .st-key-qa_panel, .st-key-growth_panel {background:#12201B !important; border-color:#22362E !important;}
.empty b {color:#E6F4EC !important;}
.st-key-navpanel button[kind="secondary"] {color:#E6F4EC !important;}
.st-key-navpanel button[kind="secondary"]:hover {background:#173B2D !important;}
.st-key-navpanel button[kind="primary"] {background:#173B2D !important; color:#8FF0C3 !important; border-color:#2F4A3F !important;}
.st-key-authcard {background:#111C18 !important; border:1px solid #1F332B !important;}
.stApp::before, .stApp::after {background:#12241D !important;}
.stTabs [data-baseweb="tab-list"] {background:#14261F !important;}
.stTabs [aria-selected="true"] {background:#1F3A30 !important;}
</style>
"""
DARK_WIDE_CSS = "<style>.block-container {background:transparent !important; border:none !important;}</style>"

def apply_pro_styles(wide: bool, theme: str = "light") -> None:
    st.markdown(PRO_CSS, unsafe_allow_html=True)
    if wide:
        st.markdown(DASH_CSS, unsafe_allow_html=True)
    if theme == "dark":
        st.markdown(DARK_CSS, unsafe_allow_html=True)
        if wide:
            st.markdown(DARK_WIDE_CSS, unsafe_allow_html=True)

def render_logo_top() -> None:
    st.markdown(_h(f"""<div class="logo-row">{LEAF_SVG}<div class="logo-text">FranchiseOps <span>AI</span></div></div>
        <div class="tag">Smarter Decisions. Stronger Franchises.</div>"""), unsafe_allow_html=True)

def render_brand_header(subtitle: str = "Secure Authentication Module", desc: str = "") -> None:
    key = subtitle.lower()
    icon = "✉️" if "recover" in key else "🔐" if ("welcome" in key or "log in" in key) else "👤"
    st.markdown(_h(f"""<div class="hero-icon">{icon}</div><div class="hero-title">{subtitle}</div>
        <div class="hero-sub">{desc}</div>"""), unsafe_allow_html=True)

def render_stepper(active: int, labels=("Email Verification", "Enter OTP")) -> str:
    on = ["on" if active == n else "" for n in (1, 2)]
    return _h(f"""<div class="stepper"><div class="step {on[0]}"><b>1</b>{labels[0]}</div>
        <div class="step {on[1]}"><b>2</b>{labels[1]}</div></div>""")

def render_topbar(name: str, role_label: str, title: str = "Dashboard") -> str:
    initials = "".join(w[0] for w in name.replace("_", " ").split()[:2]).upper() or "U"
    return _h(f"""<div class="topbar"><div class="sec-title" style="margin:0; font-size:1.15rem">{title}</div>
        <div class="chip"><div class="avatar">{initials}</div>{name} <span class="badge">{role_label}</span></div></div>""")

def stat_card(icon: str, bg: str, label: str, value, note: str, trend: str = "") -> str:
    arrow = {"up": "↗ ", "down": "↘ "}.get(trend, "")
    return (f'<div class="stat"><div class="ic" style="background:{bg}">{icon}</div>'
            f'<div class="lb">{label}</div><div class="vl">{value}</div><div class="dl {trend}">{arrow}{note}</div></div>')

def render_sidebar(pages) -> None:
    """Left navigation panel shown on dashboard pages only."""
    icons = {"Dashboard": ":material/space_dashboard:", "Profile": ":material/person:", "Settings": ":material/settings:"}
    try:
        _c = get_db_connection(); _c.execute("SELECT 1"); _c.close(); _ok = True
    except Exception:
        _ok = False
    with st.container(key="navpanel"):
        st.markdown(_h(f'<div class="logo-row" style="justify-content:flex-start">{LEAF_LIGHT}<div class="logo-text" style="font-size:1.3rem">FranchiseOps <span>AI</span></div></div>'), unsafe_allow_html=True)
        st.write("")
        current = st.session_state.get("dash_page", "Dashboard")
        for pg in pages:
            if st.button(pg, key=f"nav_{pg}", icon=icons[pg], width="stretch",
                         type="primary" if current == pg else "secondary"):
                st.session_state.dash_page = pg
                st.rerun()
        with st.container(key="nav_bottom"):
            st.markdown(f'<div class="side-status"><span class="dot" style="background:{"#18A86B" if _ok else "#D9383A"}"></span>{"Database connected" if _ok else "Database unavailable"}</div><div class="side-tag">Better Operations.<br>Bigger Dreams.</div>', unsafe_allow_html=True)
            if st.button("Log Out", key="nav_logout", icon=":material/logout:", width="stretch"):
                st.session_state.jwt_token = None
                st.session_state.current_user = None
                st.session_state.dash_page = "Dashboard"
                navigate_to("Login")

def render_footer() -> None:
    st.markdown('<div class="foot">© 2026 FranchiseOps AI. All rights reserved.</div>', unsafe_allow_html=True)

def render_login_page() -> None:
    render_brand_header("Welcome Back", "Sign in to your franchise account to continue.")

    st.session_state.setdefault("login_mode", "user")
    lc1, lc2 = st.columns(2)
    if lc1.button("👤  User Login", key="lm_user", width="stretch",
                  type="primary" if st.session_state.login_mode == "user" else "secondary"):
        st.session_state.login_mode = "user"; st.rerun()
    if lc2.button("🛡️  Admin Portal", key="lm_admin", width="stretch",
                  type="primary" if st.session_state.login_mode == "admin" else "secondary"):
        st.session_state.login_mode = "admin"; st.rerun()
    mode = st.session_state.login_mode
    st.write("")

    if mode == "user":
        with st.form("user_login_form"):
            identifier = st.text_input("Username or Email", placeholder="e.g. store_manager or user@franchise.com")
            password = st.text_input("Password", type="password", placeholder="Enter your password")
            submitted = st.form_submit_button("Sign In  →", type="primary", width="stretch")

        if submitted:
            ident_clean = identifier.strip()
            if not ident_clean or not password:
                st.error("Please enter both your username/email and password.")
            else:
                conn = get_db_connection()
                try:
                    user = conn.execute(
                        "SELECT id, username, email, password_hash, role, status, token_version FROM users WHERE username = ? OR email = ?",
                        (ident_clean, ident_clean.lower())
                    ).fetchone()

                    if not user or not verify_password(password, user["password_hash"]):
                        st.error("Invalid credentials. Please verify your username/email and password.")
                    elif user["status"] != "active":
                        st.error("This account is currently deactivated. Please contact support.")
                    else:
                        hours = 1
                        token = create_jwt_token(
                            user_id=user["id"],
                            username=user["username"],
                            role=user["role"],
                            token_version=user["token_version"],
                            hours=hours
                        )
                        st.session_state.jwt_token = token
                        st.session_state.current_user = {
                            "id": user["id"],
                            "username": user["username"],
                            "email": user["email"],
                            "role": user["role"]
                        }
                        st.success(f"Welcome back, {user['username']}!")
                        navigate_to("UserDashboard")
                finally:
                    conn.close()

        col_signup, col_forgot = st.columns(2)
        with col_signup:
            if st.button("Create Account", width="stretch"):
                navigate_to("Signup")
        with col_forgot:
            if st.button("Forgot Password?", width="stretch"):
                navigate_to("ForgotPassword")

    if mode == "admin":
        st.markdown("<p style='font-size:0.85rem; color:#68746D;'>Administrative access requires server-side authorized credentials.</p>", unsafe_allow_html=True)
        with st.form("admin_login_form"):
            admin_user = st.text_input("Admin Username", value="admin")
            admin_pass = st.text_input("Admin Password", type="password", placeholder="Enter admin password")
            admin_submit = st.form_submit_button("Authenticate as Admin  →", type="primary", width="stretch")

        if admin_submit:
            if not ADMIN_PASSWORD:
                st.error("Admin login is disabled. Add ADMIN_PASSWORD to Colab Secrets and restart the app.")
            elif admin_user.strip() == ADMIN_USERNAME and hmac.compare_digest(admin_pass, ADMIN_PASSWORD):
                # Create dedicated admin JWT
                admin_token = create_jwt_token(
                    user_id=0,
                    username=ADMIN_USERNAME,
                    role="admin",
                    token_version=1,
                    hours=2
                )
                # Overwrite sub to match server-side admin check
                payload = jwt.decode(admin_token, JWT_SECRET, algorithms=["HS256"])
                payload["sub"] = "admin_0"
                admin_token = jwt.encode(payload, JWT_SECRET, algorithm="HS256")

                st.session_state.jwt_token = admin_token
                st.session_state.current_user = {
                    "id": 0,
                    "username": ADMIN_USERNAME,
                    "email": "admin@franchiseops.internal",
                    "role": "admin"
                }
                st.success("Admin authenticated successfully.")
                navigate_to("AdminDashboard")
            else:
                st.error("Admin authentication failed. Unauthorized credentials.")
    st.markdown('<div class="secure">🔒 Your information is secure and will not be shared.</div>', unsafe_allow_html=True)

# ==============================================================================
# 9. PAGE: SIGNUP
# ==============================================================================
def render_signup_page() -> None:
    render_brand_header("Create Account", "Register a new FranchiseOps AI account.")

    with st.form("signup_form"):
        username = st.text_input("Choose Username", placeholder="e.g. apex_franchise_01")
        email = st.text_input("Official Email Address", placeholder="e.g. manager@store.com")

        col_p1, col_p2 = st.columns(2)
        with col_p1:
            pw = st.text_input("Password", type="password", placeholder="Min 8 chars, Aa, 1, @")
        with col_p2:
            cpw = st.text_input("Confirm Password", type="password", placeholder="Re-enter password")

        q_keys = list(SECURITY_QUESTIONS.keys())
        selected_q_key = st.selectbox(
            "Security Question (used for account recovery)",
            options=q_keys,
            format_func=lambda k: SECURITY_QUESTIONS[k]
        )
        sec_answer = st.text_input("Security Answer", placeholder="Your confidential answer")

        submitted = st.form_submit_button("Create Account", type="primary", width="stretch")

    if submitted:
        u_err = validate_username(username)
        e_err = validate_email(email)
        p_err = validate_password_strength(pw)

        if u_err:
            st.error(u_err)
        elif e_err:
            st.error(e_err)
        elif p_err:
            st.error(p_err)
        elif pw != cpw:
            st.error("Passwords do not match.")
        elif not sec_answer.strip():
            st.error("Security answer cannot be empty.")
        else:
            norm_u = username.strip()
            norm_e = email.strip().lower()
            conn = get_db_connection()
            try:
                # Check for existing user
                existing = conn.execute(
                    "SELECT username, email FROM users WHERE username = ? OR email = ?",
                    (norm_u, norm_e)
                ).fetchone()

                if existing:
                    if existing["username"].lower() == norm_u.lower():
                        st.error("This username is already taken. Please select another.")
                    else:
                        st.error("An account with this email address already exists.")
                else:
                    pw_hash = hash_password(pw)
                    ans_hash = hash_security_answer(sec_answer)
                    created_at = datetime.now(timezone.utc).isoformat()

                    with conn:
                        conn.execute("""
                            INSERT INTO users (username, email, password_hash, question_id, answer_hash, role, created_at, status, token_version)
                            VALUES (?, ?, ?, ?, ?, 'user', ?, 'active', 1)
                        """, (norm_u, norm_e, pw_hash, selected_q_key, ans_hash, created_at))

                    st.success("Account created successfully! You can now log in.")
                    time.sleep(1)
                    navigate_to("Login")
            finally:
                conn.close()

    st.write("")
    if st.button("&larr; Back to Login", width="stretch"):
        navigate_to("Login")

# ==============================================================================
# 10. PAGE: FORGOT PASSWORD (SECURITY QUESTION & EMAIL OTP)
# ==============================================================================
def render_forgot_password_page() -> None:
    render_brand_header("Account Recovery", "Recover access with a security question or an email code.")

    st.session_state.setdefault("rec_method", "otp")
    mc1, mc2 = st.columns(2)
    if mc1.button("🔑  Security Question", key="m_q", width="stretch",
                  type="primary" if st.session_state.rec_method == "q" else "secondary"):
        st.session_state.rec_method = "q"; st.rerun()
    if mc2.button("✉️  Email OTP", key="m_otp", width="stretch",
                  type="primary" if st.session_state.rec_method == "otp" else "secondary"):
        st.session_state.rec_method = "otp"; st.rerun()
    method = st.session_state.rec_method
    stepper_ph = st.empty()

    # ------------------ Method 1: Security Question ------------------
    if method == "q":
        st.markdown("<p style='font-size:0.85rem; color:#68746D;'>Recover your account using the security question configured during registration.</p>", unsafe_allow_html=True)

        ident = st.text_input("Enter your Username or Email", key="sq_ident", placeholder="Username or email")

        if st.button("Find Security Question", key="btn_find_q", width="stretch"):
            if not ident.strip():
                st.error("Please enter your registered username or email.")
            else:
                conn = get_db_connection()
                try:
                    u = conn.execute(
                        "SELECT id, username, email, question_id, answer_hash FROM users WHERE (username = ? OR email = ?) AND status = 'active'",
                        (ident.strip(), ident.strip().lower())
                    ).fetchone()
                    if u:
                        st.session_state.security_q_user = {
                            "id": u["id"],
                            "username": u["username"],
                            "email": u["email"],
                            "question_id": u["question_id"],
                            "answer_hash": u["answer_hash"],
                            "attempts": 0
                        }
                    else:
                        st.error("No active account found matching those details.")
                finally:
                    conn.close()

        if st.session_state.security_q_user:
            u_info = st.session_state.security_q_user
            q_text = SECURITY_QUESTIONS.get(u_info["question_id"], "Security question")
            st.info(f"**Security Question:** {q_text}")

            with st.form("form_sec_answer"):
                ans_input = st.text_input("Your Answer", placeholder="Type your answer")
                new_pw = st.text_input("New Password", type="password", placeholder="Min 8 chars, Aa, 1, @")
                c_new_pw = st.text_input("Confirm New Password", type="password", placeholder="Re-enter new password")
                reset_submit = st.form_submit_button("Reset Password", type="primary", width="stretch")

            if reset_submit:
                if u_info["attempts"] >= 5:
                    st.error("Maximum answer attempts exceeded. Security question reset locked.")
                elif not verify_security_answer(ans_input, u_info["answer_hash"]):
                    u_info["attempts"] += 1
                    rem = 5 - u_info["attempts"]
                    st.error(f"Incorrect security answer. {rem} attempt(s) remaining.")
                else:
                    p_err = validate_password_strength(new_pw)
                    if p_err:
                        st.error(p_err)
                    elif new_pw != c_new_pw:
                        st.error("New passwords do not match.")
                    else:
                        # Update password & increment token_version to revoke previous JWTs
                        new_h = hash_password(new_pw)
                        conn = get_db_connection()
                        try:
                            with conn:
                                conn.execute("""
                                    UPDATE users
                                    SET password_hash = ?, token_version = token_version + 1
                                    WHERE id = ?
                                """, (new_h, u_info["id"]))
                            st.session_state.security_q_user = None
                            st.success("Password successfully updated! All existing sessions revoked. Please log in.")
                            time.sleep(1.5)
                            navigate_to("Login")
                        finally:
                            conn.close()

    # ------------------ Method 2: Email OTP ------------------
    if method == "otp":
        st.markdown("<p style='font-size:0.9rem; color:#68746D; text-align:center;'>Enter your registered email address. We will send a 6-digit verification code, valid for 5 minutes.</p>", unsafe_allow_html=True)

        with st.form("form_otp_request"):
            email_input = st.text_input("Registered Email Address", key="otp_email_req", placeholder="name@example.com")
            send_clicked = st.form_submit_button("Send Verification Code", type="primary", width="stretch")

        if send_clicked:
            if not email_input.strip():
                st.error("Please enter your registered email address.")
            else:
                with st.spinner("Sending verification code..."):
                    ok, msg = request_password_reset_otp(email_input)
                if ok:
                    st.session_state.reset_authorized_email = email_input.strip().lower()
                    st.success(msg)
                else:
                    st.session_state.reset_authorized_email = None
                    st.error(msg)

        if st.session_state.reset_authorized_email:
            st.markdown(f"<div class='otp-card'>Code sent to <b>{st.session_state.reset_authorized_email}</b>. Check your inbox (and spam folder).</div>", unsafe_allow_html=True)

            with st.form("form_otp_verify"):
                otp_code = st.text_input("Enter 6-Digit OTP", max_chars=6, placeholder="123456")
                otp_new_pw = st.text_input("New Password", type="password", placeholder="Min 8 chars, Aa, 1, @")
                otp_c_pw = st.text_input("Confirm New Password", type="password", placeholder="Confirm new password")
                otp_submit = st.form_submit_button("Verify OTP & Update Password", type="primary", width="stretch")

            if otp_submit:
                p_err = validate_password_strength(otp_new_pw)
                if p_err:
                    st.error(p_err)
                elif otp_new_pw != otp_c_pw:
                    st.error("Passwords do not match.")
                else:
                    with st.spinner("Verifying OTP..."):
                        valid, v_msg = verify_and_consume_otp(st.session_state.reset_authorized_email, otp_code)
                    if not valid:
                        st.error(v_msg)
                    else:
                        # Update password & invalidate all previous JWT sessions
                        new_h = hash_password(otp_new_pw)
                        conn = get_db_connection()
                        try:
                            with conn:
                                conn.execute("""
                                    UPDATE users
                                    SET password_hash = ?, token_version = token_version + 1
                                    WHERE email = ?
                                """, (new_h, st.session_state.reset_authorized_email))
                            st.session_state.reset_authorized_email = None
                            st.success("Password reset successfully! Previous sessions invalidated. Please log in.")
                            time.sleep(1.5)
                            navigate_to("Login")
                        finally:
                            conn.close()

    if method == "q":
        stepper_ph.markdown(render_stepper(2 if st.session_state.security_q_user else 1, ("Find Account", "Verify & Reset")), unsafe_allow_html=True)
    else:
        stepper_ph.markdown(render_stepper(2 if st.session_state.reset_authorized_email else 1), unsafe_allow_html=True)
    st.markdown('<div class="secure">🔒 Your information is secure and will not be shared.</div>', unsafe_allow_html=True)
    if st.button("← Back to Login Page", width="stretch"):
        navigate_to("Login")

# ==============================================================================
# 11. PAGE: USER DASHBOARD
# ==============================================================================
@st.cache_resource
def app_started_at() -> float:
    return time.time()

def _user_main(user_claims: Dict[str, Any]) -> None:
    import pandas as pd
    import altair as alt
    conn = get_db_connection()
    try:
        u = conn.execute("SELECT id, username, email, role, status, created_at, question_id FROM users WHERE id = ?",
                         (user_claims["sub"],)).fetchone()
        total_users = conn.execute("SELECT COUNT(*) FROM users").fetchone()[0]
        active_users = conn.execute("SELECT COUNT(*) FROM users WHERE status = 'active'").fetchone()[0]
        signups = conn.execute("SELECT substr(created_at,1,10) d, COUNT(*) c FROM users GROUP BY d ORDER BY d").fetchall()
        otps = conn.execute("SELECT created_at, consumed FROM password_reset_otps WHERE email = ? ORDER BY id DESC LIMIT 3",
                            (u["email"] if u else "",)).fetchall()
    finally:
        conn.close()

    username = u["username"] if u else user_claims.get("username", "User")
    email = u["email"] if u else ""
    role_label = str(user_claims.get("role", "user")).capitalize()
    status = (u["status"] if u else "active").capitalize()
    created = u["created_at"][:19].replace("T", " ") if u else "N/A"
    try:
        exp_ts = jwt.decode(st.session_state.jwt_token, JWT_SECRET, algorithms=["HS256"]).get("exp")
    except Exception:
        exp_ts = None
    exp_dt = datetime.fromtimestamp(exp_ts, tz=timezone.utc) if exp_ts else None
    mins_left = max(0, int((exp_dt - datetime.now(timezone.utc)).total_seconds() // 60)) if exp_dt else None
    exp_txt = exp_dt.strftime("%Y-%m-%d %H:%M UTC") if exp_dt else "N/A"
    initials = "".join(w[0] for w in username.replace("_", " ").split()[:2]).upper() or "U"

    page = st.session_state.get("dash_page", "Dashboard")
    q = ""
    if page == "Dashboard":
        tc1, tc2 = st.columns([3, 1.3])
        q = tc1.text_input("Search", placeholder="🔍  Search modules...", label_visibility="collapsed", key="dash_search")
        tc2.markdown(_h(f'<div class="chip" style="justify-content:flex-end; margin-top:.35rem"><div class="avatar">{initials}</div>{username} <span class="badge">{role_label}</span></div>'), unsafe_allow_html=True)
    else:
        st.markdown(render_topbar(username, role_label, page), unsafe_allow_html=True)

    if page == "Profile":
        if st.session_state.get("flash"):
            st.success(st.session_state.pop("flash"))
        st.markdown(_h(f'<div class="hero sun"><div class="avatar xl">{initials}</div><div><h2>Hello, {username}</h2><p>Here is your profile information.</p></div></div>'), unsafe_allow_html=True)
        t_info, t_sec, t_act = st.tabs(["👤  Personal Info", "🛡️  Account Security", "🕒  Activity Log"])
        q_keys = list(SECURITY_QUESTIONS.keys())
        with t_info:
            st.markdown(_h(f"""<div class="card"><div class="sec-title">Personal Information</div>
                <div class="row"><span>Username</span><b>{username}</b></div>
                <div class="row"><span>User ID</span><b>{user_claims.get("sub", "-")}</b></div>
                <div class="row"><span>Email Address</span><b>{email}</b></div>
                <div class="row"><span>Account Status</span><span class="pill-ok">{status}</span></div>
                <div class="row"><span>Registered (UTC)</span><b>{created}</b></div>
                <div class="row"><span>Session Expires</span><b>{exp_txt}</b></div></div>"""), unsafe_allow_html=True)
            with st.expander("✏️  Edit Details"):
                with st.form("edit_profile_form"):
                    new_u = st.text_input("Username", value=username)
                    new_e = st.text_input("Email", value=email)
                    new_q = st.selectbox("Security Question", q_keys, format_func=lambda k: SECURITY_QUESTIONS[k],
                                         index=q_keys.index(u["question_id"]) if u and u["question_id"] in q_keys else 0)
                    new_a = st.text_input("New Security Answer (leave blank to keep the current one)", type="password")
                    pw_chk = st.text_input("Current Password (required to save changes)", type="password")
                    save = st.form_submit_button("Save Changes", type="primary", width="stretch")
                if save:
                    conn = get_db_connection()
                    try:
                        row = conn.execute("SELECT password_hash FROM users WHERE id = ?", (user_claims["sub"],)).fetchone()
                        u_err, e_err = validate_username(new_u), validate_email(new_e)
                        dup = conn.execute("SELECT 1 FROM users WHERE (username = ? OR email = ?) AND id != ?",
                                           (new_u.strip(), new_e.strip().lower(), user_claims["sub"])).fetchone()
                        if not pw_chk:
                            st.error("Enter your current password to save changes.")
                        elif not row or not verify_password(pw_chk, row["password_hash"]):
                            st.error("Current password is incorrect.")
                        elif u_err or e_err:
                            st.error(u_err or e_err)
                        elif dup:
                            st.error("That username or email is already used by another account.")
                        elif new_q != (u["question_id"] if u else new_q) and not new_a.strip():
                            st.error("Enter a new security answer when you change the security question.")
                        else:
                            with conn:
                                conn.execute("UPDATE users SET username = ?, email = ?, question_id = ? WHERE id = ?",
                                             (new_u.strip(), new_e.strip().lower(), new_q, user_claims["sub"]))
                                if new_a.strip():
                                    conn.execute("UPDATE users SET answer_hash = ? WHERE id = ?",
                                                 (hash_security_answer(new_a), user_claims["sub"]))
                            st.session_state.flash = "Profile updated successfully."
                            st.rerun()
                    finally:
                        conn.close()
            st.markdown('<div class="card" style="margin-top:1rem"><div class="sec-title">Profile Overview</div>'
                        '<div class="act">✅<div><b>You are all set!</b><small>Your profile is complete and your account is secure.</small></div></div></div>', unsafe_allow_html=True)
        with t_sec:
            qtext = SECURITY_QUESTIONS.get(u["question_id"], "Not set") if u else "Not set"
            st.markdown(_h(f"""<div class="card"><div class="sec-title">Account Security</div>
                <div class="row"><span>Security Question</span><b>{qtext}</b></div>
                <div class="row"><span>Password</span><b>Stored as a secure hash</b></div>
                <div class="row"><span>Other sessions</span><b>Signed out when the password changes</b></div>
                <div class="row"><span>Current session expires</span><b>{exp_txt}</b></div></div>"""), unsafe_allow_html=True)
            if st.button("🔑  Change Password", key="prof_pw", width="stretch"):
                st.session_state.dash_page = "Settings"
                st.rerun()
        with t_act:
            acts = [f'<div class="act">🆕<div>Account created<small>{created} UTC</small></div></div>']
            for o in otps:
                acts.append(f'<div class="act">✉️<div>Password reset code requested<small>{o["created_at"][:19].replace("T", " ")} UTC · {"used" if o["consumed"] else "unused"}</small></div></div>')
            acts.append(f'<div class="act">🔐<div>Signed in, session active<small>until {exp_txt}</small></div></div>')
            st.markdown('<div class="card"><div class="sec-title">Recent Activity</div>' + "".join(acts) + '</div>', unsafe_allow_html=True)
        return

    if page == "Settings":
        st.markdown(_h('<div class="hero night"><div style="font-size:2.6rem">⚙️</div><div><h2>Settings</h2><p>Manage your preferences and security.</p></div></div>'), unsafe_allow_html=True)
        t_gen, t_pw, t_app = st.tabs(["⚙️  General", "🔒  Security", "🎨  Appearance"])
        with t_gen:
            st.markdown(_h(f"""<div class="card"><div class="sec-title">Account Settings</div>
                <div class="row"><span>Username</span><b>{username}</b></div>
                <div class="row"><span>Email</span><b>{email}</b></div>
                <div class="row"><span>Role</span><b>{role_label}</b></div></div>"""), unsafe_allow_html=True)
            st.write("")
            if st.button("✏️  Edit Profile", key="set_edit", width="stretch"):
                st.session_state.dash_page = "Profile"
                st.rerun()
        with t_pw:
            st.markdown('<div class="sec-title">Change Password</div>', unsafe_allow_html=True)
            with st.form("change_pw_form"):
                cur = st.text_input("Current Password", type="password")
                new = st.text_input("New Password", type="password", placeholder="Min 8 chars, Aa, 1, @")
                conf = st.text_input("Confirm New Password", type="password")
                go = st.form_submit_button("Update Password", type="primary", width="stretch")
            if go:
                conn = get_db_connection()
                try:
                    row = conn.execute("SELECT password_hash FROM users WHERE id = ?", (user_claims["sub"],)).fetchone()
                    err = validate_password_strength(new) if new else None
                    if not cur or not new or not conf:
                        st.error("All fields are required.")
                    elif not row or not verify_password(cur, row["password_hash"]):
                        st.error("Current password is incorrect.")
                    elif err:
                        st.error(err)
                    elif new != conf:
                        st.error("New passwords do not match.")
                    else:
                        with conn:
                            conn.execute("UPDATE users SET password_hash = ?, token_version = token_version + 1 WHERE id = ?",
                                         (hash_password(new), user_claims["sub"]))
                        st.success("Password updated. Please log in again.")
                        time.sleep(1.2)
                        st.session_state.jwt_token = None
                        st.session_state.current_user = None
                        st.session_state.dash_page = "Dashboard"
                        navigate_to("Login")
                finally:
                    conn.close()
        with t_app:
            st.markdown('<div class="sec-title">Theme</div>', unsafe_allow_html=True)
            _cur = st.session_state.get("theme", "light")
            _pick = st.radio("Theme", ["Light", "Dark"], index=1 if _cur == "dark" else 0, horizontal=True, label_visibility="collapsed")
            if _pick.lower() != _cur:
                st.session_state.theme = _pick.lower()
                st.rerun()
        return

    st.markdown(f'<div class="welcome"><h1>Welcome back, {username}!</h1><p>Here is the live status of your franchise operations today.</p></div>', unsafe_allow_html=True)
    up = int(time.time() - app_started_at())
    ym = datetime.now(timezone.utc).strftime("%Y-%m")
    new_this_month = sum(r["c"] for r in signups if r["d"][:7] == ym)
    st.markdown('<div class="cards">' + "".join([
        stat_card("🏪", "#E3F2EA", "Total Franchises", 15, "+2 this month", "up"),
        stat_card("👥", "#E5EEFB", "Active Users", active_users, f"+{new_this_month} this month", "up"),
        stat_card("✅", "#EFE6FA", "Pending Tasks", 36, "-8% this week", "down"),
        stat_card("📊", "#FDF1DC", "App Uptime", f"{up // 3600}h {(up % 3600) // 60}m", "Since server start"),
    ]) + '</div>', unsafe_allow_html=True)
    st.caption("Franchise and task figures are sample data. Users and uptime are live.")

    gl, gr = st.columns([2.2, 1])
    with gl:
        with st.container(key="growth_panel"):
            st.markdown('<div class="sec-title">📈 Franchise Growth</div>', unsafe_allow_html=True)
            span = st.selectbox("Period", ["Last 3 Months", "Last 6 Months", "Last 12 Months"], index=1,
                                label_visibility="collapsed", key="growth_span")
            n = int(span.split()[1]); now = datetime.now(timezone.utc); labels = []
            for k in range(n - 1, -1, -1):
                mm = now.month - 1 - k
                labels.append(datetime(now.year + mm // 12, mm % 12 + 1, 1).strftime("%b"))
            sample = [3, 4, 5, 6, 7, 8.5, 10, 11, 12.5, 13.5, 14.5, 15]  # sample data for preview
            df = pd.DataFrame({"Month": labels, "Franchises": sample[-n:]})
            chart = alt.Chart(df).mark_area(line={"color": "#176B45"}, point={"color": "#176B45"}, color="#CDEBDB", opacity=0.7).encode(
                x=alt.X("Month:N", sort=labels, axis=alt.Axis(labelAngle=0, title=None)),
                y=alt.Y("Franchises:Q", scale=alt.Scale(domain=[0, 20]), axis=alt.Axis(title=None, tickMinStep=5)),
            ).properties(height=230)
            st.altair_chart(chart, width="stretch")
            st.caption("Sample data for preview. Real values appear once franchises are added.")
    with gr:
        with st.container(key="qa_panel"):
            st.markdown('<div class="sec-title">⚡ Quick Actions</div>', unsafe_allow_html=True)
            for label, target in [("👤  My Profile", "Profile"), ("⚙️  Account Settings", "Settings"), ("🔑  Change Password", "Settings")]:
                if st.button(label, key=f"qa_{label}", width="stretch"):
                    st.session_state.dash_page = target
                    st.rerun()
            st.button("➕  Add Franchise · Milestone 2+", key="qa_add", width="stretch", disabled=True)

    st.markdown('<div class="copilot" style="display:flex; justify-content:space-between; align-items:center; gap:1rem">'
                '<div><h3>✨ AI Copilot</h3><p>Get instant insights, answers and recommendations with AI.</p></div>'
                '<span class="pillbtn">Milestone 2+</span></div>', unsafe_allow_html=True)

    mods = [("🕸️", "Knowledge Graph", "Explore connected data across franchises, regions and more."),
            ("🧊", "Digital Twin", "Simulate and optimize your franchise network."),
            ("⚠️", "Anomaly Scanner", "Detect unusual patterns and prevent issues early."),
            ("🗄️", "Data Feed Center", "Access and manage your franchise data feeds.")]
    shown = [m for m in mods if q.strip().lower() in (m[1] + " " + m[2]).lower()]
    if shown:
        st.markdown('<div class="mods">' + "".join(
            f'<div class="card"><div style="font-size:1.4rem">{i}</div><h4>{t}</h4><p>{d}</p><span class="pillbtn" style="display:inline-block; margin-top:.6rem">Milestone 2+</span></div>'
            for i, t, d in shown) + '</div>', unsafe_allow_html=True)
    else:
        st.info("No modules match your search.")

    cl, cr = st.columns([1.6, 1])
    with cl:
        st.markdown('<div class="sec-title">New Sign-ups by Day</div>', unsafe_allow_html=True)
        st.bar_chart(pd.DataFrame({"Sign-ups": [r["c"] for r in signups]}, index=[r["d"] for r in signups]),
                     color="#18A86B", height=220)
    with cr:
        acts = [f'<div class="act">🆕<div>Account created<small>{created} UTC</small></div></div>']
        for o in otps:
            acts.append(f'<div class="act">✉️<div>Password reset code requested<small>{o["created_at"][:19].replace("T", " ")} UTC · {"used" if o["consumed"] else "unused"}</small></div></div>')
        acts.append(f'<div class="act">🔐<div>Signed in, session active<small>until {exp_txt}</small></div></div>')
        st.markdown('<div class="card"><div class="sec-title">Recent Activity</div>' + "".join(acts) + '</div>', unsafe_allow_html=True)

def _admin_main(admin_claims: Dict[str, Any]) -> None:
    if st.session_state.get("dash_page", "Dashboard") == "Settings":
        st.markdown(render_topbar("Administrator", "Admin", "Settings"), unsafe_allow_html=True)
        st.markdown('<div class="sec-title">Appearance</div>', unsafe_allow_html=True)
        _cur = st.session_state.get("theme", "light")
        _pick = st.radio("Theme", ["Light", "Dark"], index=1 if _cur == "dark" else 0, horizontal=True, label_visibility="collapsed")
        if _pick.lower() != _cur:
            st.session_state.theme = _pick.lower()
            st.rerun()
        st.markdown(_h(f"""<div class="card" style="margin-top:1rem"><div class="sec-title">Admin Access</div>
            <div class="row"><span>Admin username</span><b>{ADMIN_USERNAME}</b></div>
            <div class="row"><span>Credentials source</span><b>Colab Secrets</b></div>
            <div class="row"><span>Session length</span><b>2 hours</b></div></div>"""), unsafe_allow_html=True)
        return
    st.markdown(render_topbar("Administrator", "Admin", "Admin Dashboard"), unsafe_allow_html=True)
    st.markdown('<div class="welcome"><h1>Executive Admin Portal</h1><p>Manage registered users and monitor authentication activity.</p></div>', unsafe_allow_html=True)

    conn = get_db_connection()
    try:
        users = conn.execute(
            "SELECT id, username, email, role, created_at, status, token_version FROM users ORDER BY id ASC"
        ).fetchall()
        otp_stats = conn.execute(
            "SELECT COUNT(*) as total, SUM(consumed) as consumed_cnt FROM password_reset_otps"
        ).fetchone()
    finally:
        conn.close()

    total_users = len(users)
    active_users = sum(1 for u in users if u["status"] == "active")
    inactive_users = total_users - active_users
    total_otps = otp_stats["total"] if otp_stats and otp_stats["total"] else 0

    st.markdown('<div class="cards">' + "".join([
        stat_card("👥", "#E3F2EA", "Total Users", total_users, "Registered accounts"),
        stat_card("✅", "#E5EEFB", "Active Users", active_users, "Can sign in"),
        stat_card("⏸️", "#FDF1DC", "Inactive", inactive_users, "Deactivated"),
        stat_card("✉️", "#EFE6FA", "OTP Requests", total_otps, "Password resets"),
    ]) + '</div>', unsafe_allow_html=True)

    st.write("---")
    st.markdown("### Registered User Directory")
    st.caption("Confidential administrative view. Passwords, hashes, and security answers are strictly excluded.")

    search_q = st.text_input("Filter users by username or email", placeholder="Type to search...")

    filtered_users = []
    for u in users:
        if search_q.strip().lower() in u["username"].lower() or search_q.strip().lower() in u["email"].lower():
            filtered_users.append({
                "User ID": u["id"],
                "Username": u["username"],
                "Email": u["email"],
                "Role": u["role"],
                "Status": u["status"].capitalize(),
                "Token Ver": u["token_version"],
                "Registered At": u["created_at"][:19].replace("T", " ")
            })

    if filtered_users:
        st.dataframe(filtered_users, width="stretch", hide_index=True)
    else:
        st.info("No registered users match your search query.")

# ==============================================================================
# 13. MAIN APPLICATION ROUTER
# ==============================================================================
def render_user_dashboard(user_claims: Dict[str, Any]) -> None:
    nav, main = st.columns([1, 4.2], gap="medium")
    with nav:
        render_sidebar(["Dashboard", "Profile", "Settings"])
    with main:
        _user_main(user_claims)

def render_admin_dashboard(admin_claims: Dict[str, Any]) -> None:
    nav, main = st.columns([1, 4.2], gap="medium")
    with nav:
        render_sidebar(["Dashboard", "Settings"])
    with main:
        _admin_main(admin_claims)

def main() -> None:
    st.set_page_config(
        page_title="FranchiseOps AI — Authentication",
        page_icon="🏢",
        layout="centered",
        initial_sidebar_state="expanded"
    )
    apply_custom_styles()
    init_session_state()
    current_page = st.session_state.page
    apply_pro_styles(current_page in ("UserDashboard", "AdminDashboard"), st.session_state.get("theme", "light"))

    # Pages that require valid JWT session
    if current_page in ("UserDashboard", "AdminDashboard"):
        valid, claims, err_msg = verify_jwt_token(st.session_state.jwt_token)
        if not valid or not claims:
            st.error(f"Authentication required: {err_msg}")
            time.sleep(1)
            st.session_state.jwt_token = None
            st.session_state.current_user = None
            navigate_to("Login")
            return

        if current_page == "AdminDashboard":
            if claims.get("role") != "admin":
                st.error("Access denied. Admin privileges required.")
                navigate_to("UserDashboard")
                return
            render_admin_dashboard(claims)
        else:
            render_user_dashboard(claims)

    else:
        render_logo_top()
        with st.container(key="authcard"):
            if current_page == "Signup":
                render_signup_page()
            elif current_page == "ForgotPassword":
                render_forgot_password_page()
            else:
                render_login_page()

    if current_page not in ("UserDashboard", "AdminDashboard"):
        render_footer()

if __name__ == "__main__":
    main()


Overwriting /content/franchiseops_ai/app.py


In [49]:
import subprocess
import time
import urllib.request

# Terminate existing Streamlit processes to prevent port conflicts
!pkill -f "streamlit" 2>/dev/null

print("Starting Streamlit server on port 8501...")
# Streamlit runs as a separate process and cannot read Colab Secrets itself,
# so read them here (in the notebook) and hand them over as environment variables.
import os
from google.colab import userdata

child_env = os.environ.copy()
for _name in ["JWT_SECRET", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "ADMIN_USERNAME", "ADMIN_PASSWORD"]:
    try:
        child_env[_name] = userdata.get(_name)
    except Exception:
        pass  # secret not set -> app falls back to its default / shows a clear message

# Keep the user database on Google Drive so accounts survive runtime resets.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    child_env["FRANCHISEOPS_DB_DIR"] = "/content/drive/MyDrive/FranchiseOps_AI_data"
    print("Database folder: Google Drive > FranchiseOps_AI_data (accounts will be kept)")
except Exception as _e:
    print("Drive not mounted - accounts will be lost when the runtime resets.", type(_e).__name__)

streamlit_proc = subprocess.Popen(
    ["streamlit", "run", f"{APP_DIR}/app.py", "--server.port", "8501", "--server.headless", "true", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--theme.base", "light", "--theme.primaryColor", "#176B45", "--theme.backgroundColor", "#F6FAF8",
     "--theme.secondaryBackgroundColor", "#FFFFFF", "--theme.textColor", "#17231D"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=child_env
)

# Health check loop
server_started = False
for i in range(15):
    time.sleep(1)
    try:
        resp = urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2)
        if resp.getcode() == 200:
            server_started = True
            break
    except Exception:
        pass

if server_started:
    print(" Streamlit is active and healthy on http://localhost:8501!")
else:
    print(" Streamlit is starting up (or running in background). Continuing to ngrok tunnel...")


^C
Starting Streamlit server on port 8501...
Drive not mounted - accounts will be lost when the runtime resets. MessageError
 Streamlit is active and healthy on http://localhost:8501!


In [50]:
import logging
logging.getLogger("pyngrok").setLevel(logging.CRITICAL)  # hide harmless agent-restart messages
from pyngrok import ngrok, conf
import subprocess, time

def stop_local_ngrok():
    """Close every tunnel and ngrok process that belongs to THIS Colab runtime."""
    try:
        for t in ngrok.get_tunnels():
            ngrok.disconnect(t.public_url)
    except Exception:
        pass
    try:
        ngrok.kill()
    except Exception:
        pass
    subprocess.run(["pkill", "-f", "ngrok"], check=False)
    time.sleep(2)

def launch_ngrok_tunnel(max_wait_seconds: int = 150):
    token = None
    if COLAB_AVAILABLE:
        try:
            token = userdata.get("NGROK_AUTHTOKEN")
        except Exception:
            pass
    if not token:
        token = os.environ.get("NGROK_AUTHTOKEN")

    if not token:
        print("[!] ERROR: 'NGROK_AUTHTOKEN' was not found in Google Colab Secrets.")
        print("Please add NGROK_AUTHTOKEN in the Secrets (key icon) tab and re-run this cell.")
        return None

    stop_local_ngrok()
    conf.get_default().auth_token = str(token).strip()

    # ERR_NGROK_334 = your free ngrok address is still held by an older session
    # (another tab, your laptop, or a runtime that just closed). Old sessions
    # expire on their own after a short time, so retry for a while.
    deadline = time.time() + max_wait_seconds
    attempt = 0
    while True:
        attempt += 1
        try:
            tunnel = ngrok.connect(8501, "http")
            break
        except Exception as e:
            if "ERR_NGROK_334" in str(e) and time.time() < deadline:
                print(f"[!] Attempt {attempt}: your ngrok address is still in use by an older session. Retrying in 15s...")
                stop_local_ngrok()
                time.sleep(13)
                continue
            if "ERR_NGROK_334" in str(e):
                print("[!] The ngrok address is STILL in use by another session.")
                print("Fix: open https://dashboard.ngrok.com -> Agents (or Endpoints) -> stop every running one,")
                print("also close ngrok/streamlit terminals on your laptop and other Colab tabs, then re-run this cell.")
                return None
            raise

    public_url = tunnel.public_url
    print("=" * 65)
    print(" FRANCHISEOPS AI IS LIVE!")
    print(f" Public URL: {public_url}")
    print("=" * 65)
    return public_url

public_url = launch_ngrok_tunnel()


 FRANCHISEOPS AI IS LIVE!
 Public URL: https://hardhead-justice-record.ngrok-free.dev


In [51]:
import jwt  # pip install PyJWT
from datetime import datetime, timedelta, timezone

# ── JWT Configuration ──────────────────────────────────────────────────────
JWT_SECRET_KEY = "FranchiseOpsAI_JWT_SecretKey_Infosys_2026_Batch4"
JWT_ALGORITHM  = "HS256"
JWT_EXPIRY_MINUTES = 60          # 1 hour for normal login
JWT_REMEMBER_DAYS  = 7           # 7 days for "remember me"

def create_jwt_token(user_id: int, username: str, role: str,
                     token_version: int, remember_me: bool = False) -> str:
    """Create a signed HS256 JWT with server-side revocation support."""
    now = datetime.now(timezone.utc)
    if remember_me:
        expiry = now + timedelta(days=JWT_REMEMBER_DAYS)
    else:
        expiry = now + timedelta(minutes=JWT_EXPIRY_MINUTES)

    payload = {
        "sub"   : str(user_id),
        "usr"   : username,
        "role"  : role,
        "ver"   : token_version,   # server-side revocation version
        "iat"   : now,
        "exp"   : expiry,
    }
    return jwt.encode(payload, JWT_SECRET_KEY, algorithm=JWT_ALGORITHM)


def verify_jwt_token(token: str, db_path: str = DB_PATH) -> Tuple[bool, Optional[Dict[str, Any]], str]:
    """
    Verify JWT signature + expiry + server-side token version.
    Returns (is_valid, claims_dict, message).
    """
    try:
        claims = jwt.decode(token, JWT_SECRET_KEY, algorithms=[JWT_ALGORITHM])
    except jwt.ExpiredSignatureError:
        return False, None, "Session expired. Please log in again."
    except jwt.InvalidTokenError as e:
        return False, None, f"Invalid token: {e}"

    # Server-side revocation check — compare token version with DB
    try:
        conn = sqlite3.connect(db_path)
        row = conn.execute(
            "SELECT token_version, status FROM users WHERE id = ?",
            (int(claims["sub"]),)
        ).fetchone()
        conn.close()
    except Exception:
        return False, None, "Database error during token validation."

    if row is None:
        return False, None, "User account not found."
    if row[1] != "active":
        return False, None, "Account is suspended."
    if row[0] != claims.get("ver"):
        return False, None, "Session invalidated. Please log in again."

    return True, claims, "Token valid."


print("JWT HS256 token utilities configured.")


JWT HS256 token utilities configured.
